# 🧮 Math LLM — Build a Language Model for Mathematics from Scratch

This notebook walks you through building, training, and evaluating a
**decoder-only GPT-style Transformer** that solves mathematical problems.

**Everything is implemented from scratch** using PyTorch primitives.
No pretrained weights. No LLM APIs.

This notebook **clones the actual project repository** rather than
re-typing copies of each module inline. That is a deliberate fix: an
earlier version of this notebook embedded hand-copied snapshots of
`dataset.py`/`utils.py` that drifted out of sync with the real modules
(they even duplicated a since-fixed off-by-one bug in the accuracy
metric). Cloning the repo means the notebook and the CLI scripts
(`train.py`, `evaluate.py`, `generate.py`) always run the exact same code.

---
**Before running:** `Runtime → Change runtime type → T4 GPU`

**Estimated time:** ~30–60 minutes on T4 for the baseline (20 K steps, Stage 1),
plus a one-time ~2.3 GB dataset download (a few minutes).

## Step 1 — Clone the repository

In [ ]:
# Set GIT_REF to the branch/tag/commit you want to run. Defaults to 'main'.
GIT_REF = 'main'

import os
REPO_URL = 'https://github.com/sinor77/math-llm'
REPO_PATH = '/content/math-llm'

if not os.path.isdir(REPO_PATH):
    !git clone -q {REPO_URL} {REPO_PATH}
!git -C {REPO_PATH} fetch -q origin {GIT_REF}
!git -C {REPO_PATH} checkout -q {GIT_REF}
!git -C {REPO_PATH} pull -q origin {GIT_REF} || true
print(f'Repository ready at {REPO_PATH}  (ref={GIT_REF})')


## Step 2 — Install dependencies and set up working directory

In [ ]:
# autoreload makes Jupyter/Colab re-import project modules (generate.py,
# dataset.py, ...) whenever their FILE CONTENT CHANGES -- without this,
# once a module is imported once in this kernel, re-running the clone-repo
# cell's `git pull` updates the files on disk but this kernel keeps using
# the OLD in-memory module (Python only executes a module's top-level code
# on its first import per process). That silently made every later fix
# pulled from GitHub invisible until a full Runtime > Restart. autoreload 2
# checks file mtimes before every cell and reloads anything changed,
# including modules already imported earlier in this same session.
%load_ext autoreload
%autoreload 2

import sys, os
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)

!pip install -q -r requirements.txt

# ── Persist checkpoints to Google Drive ─────────────────────────────────
# Colab's local disk (everything under /content, including checkpoints/)
# is wiped on EVERY runtime restart or disconnect -- not just ones you
# choose (e.g. to pick up a pulled fix), but Colab's own idle/timeout
# disconnects too. Writing checkpoints straight to Drive during training
# (instead of only copying them there manually AFTER training finishes,
# which doesn't help if a restart happens mid-run) makes a run resumable
# across restarts, not just within one still-running session.
USE_DRIVE = True

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        CHECKPOINT_DIR = '/content/drive/MyDrive/math-llm/checkpoints'
        print(f'Checkpoints will persist to Google Drive: {CHECKPOINT_DIR}')
    except ImportError:
        print('Not running in Colab (or Drive unavailable) — falling back '
              'to local, non-persistent storage.')
        USE_DRIVE = False

if not USE_DRIVE:
    CHECKPOINT_DIR = 'checkpoints'
    print('WARNING: checkpoints are LOCAL ONLY and will be LOST on the next '
          'runtime restart/disconnect. Set USE_DRIVE = True above to '
          'persist across restarts (recommended for any long training run).')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('data_cache', exist_ok=True)
print(f'Working directory : {os.getcwd()}')
print(f'Checkpoint dir    : {CHECKPOINT_DIR}')


## Step 3 — Verify GPU

In [ ]:
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    DEVICE = 'cuda'
else:
    print('WARNING: No GPU. Training will be very slow.')
    DEVICE = 'cpu'
print(f'Device          : {DEVICE}')


## Step 4 — Configuration
Change `MODEL_NAME`, `TRAIN_PRESET`, and `DATA_STAGE` here and re-run this one cell.
All modules (`config.py`, `tokenizer.py`, `attention.py`, `transformer.py`,
`model.py`, `dataset.py`, `utils.py`, `train.py`, `evaluate.py`, `generate.py`)
are imported directly from the cloned repository — nothing is re-typed here.

In [ ]:
from config import get_model_config, get_train_config, get_curriculum, DataConfig

# ── Model size ────────────────────────────────────────────────────────────
# 'tiny-1M' → smoke-test in minutes
# 'small-2M' → default starting point
# 'medium-5M' → try this if accuracy is below target
# 'large-10M' → push for best accuracy
MODEL_NAME = 'small-2M'

# ── Training budget ───────────────────────────────────────────────────────
# 'fast'    →  5 000 steps  (~5 min T4)   — just confirms the pipeline runs
# 'default' → 20 000 steps  (~30 min T4)  — proper baseline
# 'thorough'→ 50 000 steps  (~75 min T4)  — serious accuracy attempt
# Ignored if CURRICULUM is not 'none' — the curriculum preset sets its own
# total step budget instead (see below).
TRAIN_PRESET = 'default'

# ── Dataset source ────────────────────────────────────────────────────────
# 'real'      → the actual DeepMind Mathematics Dataset (Saxton et al.
#               2019): full difficulty (numbers up to 8 digits, decimals,
#               negatives). Hard to reach 95%+ exact-answer accuracy on —
#               that's expected, not a bug (see README's Results section).
# 'generated' → a controlled, Python-generated arithmetic benchmark
#               bounded to GEN_MIN_DIGITS..GEN_MAX_DIGITS operand digits.
#               Every equation is still real, exact arithmetic (computed
#               once here to build the corpus) — this is an explicit,
#               clearly-labelled choice, never silently substituted, and
#               source_info always says "GENERATED" (never "REAL") when
#               this path is used. Use this if your goal is a bounded
#               benchmark you can realistically push toward 95%+.
DATASET_SOURCE = 'generated'
GEN_MIN_DIGITS = 1
GEN_MAX_DIGITS = 4

# Operators to include. A measured run with all four (+,-,*,/) gave 33.6%
# overall (add/sub ~53%, mul/div only ~10-17%) — division adds exact-
# integer-quotient solving on top of carrying, a second problem stacked on
# the first. Excluding it here isolates whether the architecture can
# master carrying-based arithmetic (+,-,*) before tackling division
# separately. Set back to ['+','-','*','/'] once +,-,* are solid.
GEN_OPS = ['+', '-', '*']

# Write the answer's digits least-significant-first in the training text
# (e.g. 10366 -> "66301"; generate.generate_answer() reverses it back
# automatically everywhere it's used, so this is invisible outside
# training). A measured run WITHOUT this got 86.6%/89.8% on add/sub, with
# failures concentrated almost entirely in one pattern: dropping the
# leading digit whenever carrying produced a result one digit longer than
# both operands (9286+3247=12533 predicted as 2533) — generating most-
# significant-digit-first forces committing to the answer's length before
# the full carry chain is known. Independently used by
# github.com/brendanlong/math-llm's chain-of-thought approach for the
# same reason. item['answer'] and everything you print/see stays in
# normal reading order regardless of this setting.
GEN_REVERSE_ANSWER = True

# Train '*' examples as an explicit long-multiplication chain of thought
# (one partial product per nonzero digit of the second operand, then a
# running sum) instead of a single-shot final answer, e.g.
# "23 * 45" -> "23*5=115,23*40=920,115+920=1035". A measured run WITHOUT
# this got add=95.8%/sub=99.4% but mul stuck at only 20.0% even with the
# same curriculum + reverse-answer treatment: multiplying two multi-digit
# numbers has no local digit-by-digit pattern the way carrying does, so
# the model has to implicitly compute and sum cross-digit partial
# products all in one shot. This reduces it to multi-digit x single-digit
# multiplication (much easier) plus summing a short list of numbers
# (already solved for addition). item['answer'] stays just the final
# result either way; generate.generate_answer() extracts it automatically.
GEN_MUL_COT = True

# ── Curriculum: train easy -> hard by number length ─────────────────────
# A character-level model has no built-in notion of place value — it has
# to learn digit-by-digit carrying purely from examples. Training on the
# full target difficulty from step 0 leaves exact-answer accuracy low for
# a long time even as loss falls, because every digit position has to be
# right simultaneously to count as correct.
#
# Setting a curriculum instead restricts training to short numbers first,
# then progressively lifts the restriction:
#   'none'            → flat TRAIN_PRESET budget, full difficulty from step 0
#   'default'         → real-dataset ramp (verified against real data), 20 000 steps
#   'fast'            → same real-dataset ramp, 5 000 steps (smoke test)
#   'thorough'        → same real-dataset ramp, 50 000 steps
#   'digits_1_4'      → ramp for DATASET_SOURCE='generated', GEN_MAX_DIGITS=4:
#                       1 -> 2 -> 3 -> 4 digits, 20 000 steps evenly-ish split
#   'digits_1_4_long' → same ramp, but 80 000 steps (2K/4K/8K/66K) with most
#                       of the budget on the final (hardest, full-range)
#                       stage. A measured 'digits_1_4' 20K-step run left
#                       accuracy still climbing when the final stage
#                       ended — it simply hadn't had enough time on the
#                       hardest examples yet. This model trains fast
#                       enough on a T4 that spending far more budget here
#                       is cheap. THIS is the configuration to use for a
#                       real result, not 'digits_1_4'.
CURRICULUM = 'digits_1_4_long' if DATASET_SOURCE == 'generated' else 'default'

# Early-stopping patience (evals with no val-loss improvement before a
# STAGE stops early — not the whole run). The default (5 evals = 2 500
# steps of no improvement) can cut off progress prematurely in a long
# final stage if it plateaus before a later breakthrough. Raise it when
# using 'digits_1_4_long'.
PATIENCE = 15 if CURRICULUM == 'digits_1_4_long' else 5

# ── Data stage (real dataset only; ignored for DATASET_SOURCE='generated') ─
# 1 = add/sub + mul + div + mixed  (simplest, trains fastest)
# 2 = stage 1 + comparison categories
# 3 = stage 2 + algebra + probability + measurement
DATA_STAGE = 1

model_cfg = get_model_config(MODEL_NAME)
train_cfg = get_train_config(TRAIN_PRESET)
train_cfg.curriculum = get_curriculum(CURRICULUM)
train_cfg.patience = PATIENCE
train_cfg.device = DEVICE

data_cfg = DataConfig()
data_cfg.dataset_source = DATASET_SOURCE
if DATASET_SOURCE == 'generated':
    data_cfg.generated_min_digits = GEN_MIN_DIGITS
    data_cfg.generated_max_digits = GEN_MAX_DIGITS
    data_cfg.generated_ops = GEN_OPS
    data_cfg.generated_reverse_answer = GEN_REVERSE_ANSWER
    data_cfg.generated_mul_cot = GEN_MUL_COT
else:
    if DATA_STAGE == 1:   data_cfg.active_categories = data_cfg.categories_stage1
    elif DATA_STAGE == 2: data_cfg.active_categories = data_cfg.categories_stage1 + data_cfg.categories_stage2
    else:                 data_cfg.active_categories = data_cfg.categories_stage1 + data_cfg.categories_stage2 + data_cfg.categories_stage3

total_steps = sum(s['steps'] for s in train_cfg.curriculum) if train_cfg.curriculum else train_cfg.max_steps
print(f'Model      : {MODEL_NAME}')
print(f'Source     : {DATASET_SOURCE}' + (f'  (digits {GEN_MIN_DIGITS}-{GEN_MAX_DIGITS}, ops={GEN_OPS}, reverse_answer={GEN_REVERSE_ANSWER}, mul_cot={GEN_MUL_COT})' if DATASET_SOURCE == 'generated' else ''))
print(f'Curriculum : {CURRICULUM}  ({total_steps:,} steps total)  patience={PATIENCE}')
if DATASET_SOURCE != 'generated':
    print(f'Stage      : {DATA_STAGE}')
    print(f'Categories ({len(data_cfg.active_categories)}):')
    for c in data_cfg.active_categories:
        print(f'  {c}')


## Step 5 — Load the REAL dataset and verify it

This downloads (and caches — one-time, ~2.3 GB) the actual
**DeepMind Mathematics Dataset** (Saxton et al., ICLR 2019) directly from
its official source, extracts only the active categories, and splits it:

- **train / val** — drawn from the `train-easy` difficulty tier (our own
  deterministic hash split).
- **test** — the dataset authors' own `interpolate` tier: independently
  generated questions, never touched during training. Genuinely unseen
  by construction, not just by a hash bucket.

**This step does NOT silently fall back to synthetic data.** If the real
dataset cannot be downloaded or parsed, `load_split_dataset` raises
`RealDatasetLoadError` with the underlying exception and this cell stops
— that is the fix for the original bug (`Data source: synthetic`).

In [ ]:
from dataset import load_split_dataset, format_example, RealDatasetLoadError

try:
    train_items, val_items, test_items, source_info = load_split_dataset(data_cfg)
except RealDatasetLoadError as exc:
    print('\n' + '!'*60)
    print('REAL DATASET LOAD FAILED')
    print('Reason:')
    print(exc)
    print('!'*60)
    raise

assert source_info['data_source'] in ('REAL', 'GENERATED')
assert sum(source_info['overlap'].values()) == 0, 'Non-zero overlap between splits!'

print(f"\n--- 10 representative {source_info['data_source']} examples (train set) ---")
for it in train_items[:10]:
    print(f"  [{it['category']}]")
    print(f"  <Q>{it['question']}<A>{it['answer']}<EOS>")


## Step 6 — Build tokenizer and DataLoaders

In [ ]:
from dataset import build_dataloaders_from_items

# Honor the same reverse_answer setting Step 9's real training run will use
# (only meaningful for DATASET_SOURCE='generated') so this sanity check's
# sample output actually reflects what the model is trained on.
_reverse = data_cfg.generated_reverse_answer if DATASET_SOURCE == 'generated' else False

train_loader, val_loader, test_loader, tokenizer = build_dataloaders_from_items(
    train_items, val_items, test_items, model_cfg, batch_size=train_cfg.batch_size,
    reverse_answer=_reverse,
)
tokenizer.save(os.path.join(CHECKPOINT_DIR, 'tokenizer.json'))

# ── round-trip test on a REAL example ─────────────────────────────────────
sample = format_example(train_items[0], reverse_answer=_reverse)
ids    = tokenizer.encode(sample)
back   = tokenizer.decode(ids)
assert back == sample, f'Round-trip FAILED:\n  in : {sample!r}\n  out: {back!r}'
print(f'Round-trip test : PASSED')
print(f'  example : {sample}' + ('  (answer reversed — this is the training format, not the display format)' if _reverse else ''))

b0 = next(iter(train_loader))
print(f'\nBatch shape: input_ids={tuple(b0["input_ids"].shape)}')
print(f'First training example (decoded):')
print(f'  {tokenizer.decode(b0["input_ids"][0].tolist())}')
n_answer_tokens = (b0['labels'] != -100).sum().item()
assert n_answer_tokens > 0, 'FATAL: every label is -100 — nothing to train on'
print(f'Answer tokens in first batch: {n_answer_tokens} — PASSED')


## Step 7 — Build model and verify forward pass

In [ ]:
from model import MathLLM
import math

device = torch.device(DEVICE)
model  = MathLLM(model_cfg).to(device)
print(f'Trainable parameters: {model.num_parameters():,}')

model.eval()
with torch.no_grad():
    real_ids    = b0['input_ids'][:2].to(device)
    real_labels = b0['labels'][:2].to(device)
    out = model(real_ids, labels=real_labels)
print(f'Forward pass:  logits={tuple(out["logits"].shape)}  loss={out["loss"].item():.4f}')
expected_random_loss = math.log(tokenizer.vocab_size)
print(f'Expected random-init loss ≈ log({tokenizer.vocab_size}) = {expected_random_loss:.4f}')
assert out['loss'].item() < expected_random_loss * 2, 'Loss is suspiciously high'
print('Loss sanity check: PASSED')
model.train()


## Step 8 — MANDATORY tiny-overfit sanity check

Before any long training run, prove the pipeline can memorize a small,
controlled set of REAL examples. If the model cannot memorize even 60
examples, the bug is in the pipeline (tokenizer/labels/model/loss/
generation) — a bigger model or a longer run will not fix it. See
`tiny_overfit.py` in the repo for the full standalone version of this
check (with a separate generation test).

In [ ]:
from dataset import MathDataset, collate_fn
from functools import partial
from collections import defaultdict
from generate import generate_answer
from utils import token_accuracy

# Honor the same reverse_answer setting the real training run will use
# (only meaningful for DATASET_SOURCE='generated') so this sanity check
# actually exercises the same code path, not a different one.
_reverse = data_cfg.generated_reverse_answer if DATASET_SOURCE == 'generated' else False

# Stratify across categories instead of train_items[:60] -- with several
# operators active, plain insertion order puts EVERY category's examples
# together (all '+' first, then all '-', then all '*'), so a naive prefix
# slice can silently test only one operator and never exercise the
# others (e.g. multiplication's chain-of-thought format, if
# GEN_MUL_COT=True) at all.
_by_cat = defaultdict(list)
for it in train_items:
    _by_cat[it['category']].append(it)
_per_cat = max(1, 60 // max(len(_by_cat), 1))
tiny_items = []
for _cat, _items in _by_cat.items():
    tiny_items.extend(_items[:_per_cat])
print(f'Tiny-overfit sample: {len(tiny_items)} examples across categories {sorted(_by_cat.keys())}')

tiny_ds = MathDataset(tiny_items, tokenizer, max_seq_len=model_cfg.max_seq_len, reverse_answer=_reverse)
tiny_collate = partial(collate_fn, pad_id=tokenizer.pad_id)
tiny_batch = tiny_collate([tiny_ds[i] for i in range(len(tiny_ds))])
tiny_ids, tiny_labels = tiny_batch['input_ids'].to(device), tiny_batch['labels'].to(device)

tiny_model = MathLLM(model_cfg).to(device)
tiny_opt = torch.optim.AdamW(tiny_model.parameters(), lr=3e-3)
tiny_model.train()
initial_loss = None
for step in range(600):
    tiny_opt.zero_grad(set_to_none=True)
    out = tiny_model(tiny_ids, labels=tiny_labels)
    if initial_loss is None:
        initial_loss = out['loss'].item()
    out['loss'].backward()
    torch.nn.utils.clip_grad_norm_(tiny_model.parameters(), 1.0)
    tiny_opt.step()
final_loss = out['loss'].item()
final_acc  = token_accuracy(out['logits'].detach(), tiny_labels)

tiny_model.eval()
n_correct = 0
for it in tiny_items:
    pred, _ = generate_answer(tiny_model, tokenizer, it['question'], device, max_new_tokens=160, reverse_answer=_reverse)
    n_correct += int(pred.strip() == it['answer'].strip())
gen_acc = n_correct / len(tiny_items)

print(f'Initial loss          : {initial_loss:.4f}')
print(f'Final loss            : {final_loss:.4f}')
print(f'Final token accuracy  : {final_acc:.4f}')
print(f'Generation accuracy   : {gen_acc:.4f}  ({n_correct}/{len(tiny_items)})')
assert final_acc > 0.95 and gen_acc > 0.90, (
    'TINY OVERFIT TEST FAILED — do not proceed to full training. '
    'The pipeline has a bug (see tiny_overfit.py to debug offline).'
)
print('\nTINY OVERFIT TEST: PASSED — pipeline verified end-to-end.')
del tiny_model, tiny_opt  # free memory before the real training run


## Step 9 — Train
This calls the exact same `train()` function used by `python train.py` on
the command line — the training loop, optimiser, scheduler, checkpointing
and logging are implemented in `train.py`/`utils.py` (open them in the
Colab file browser on the left to read the full, commented implementation).
Watch that `loss` falls and `tok_acc` rises.

In [ ]:
from train import train as run_training
import glob

train_cfg.checkpoint_dir = CHECKPOINT_DIR

# Resume automatically if a checkpoint already exists in CHECKPOINT_DIR --
# from an earlier run of THIS cell in the same session, or (with
# USE_DRIVE=True above) from a previous session entirely, since Drive
# survives runtime restarts. Without this, every run of this cell started
# over from step 0 and discarded all previous progress even when the
# weights were sitting right there on disk.
import re as _re
_step_ckpts = sorted(
    glob.glob(os.path.join(CHECKPOINT_DIR, 'step_*.pt')),
    key=lambda p: int(_re.search(r'step_(\d+)', os.path.basename(p)).group(1)),
)
resume_from = _step_ckpts[-1] if _step_ckpts else None
if resume_from:
    print(f'Found existing checkpoint — resuming training from: {resume_from}')
else:
    print(f'No existing checkpoint found in {CHECKPOINT_DIR!r} — starting fresh.')

history = run_training(model_cfg, train_cfg, data_cfg, resume_from=resume_from)


## Step 10 — Training curves

In [ ]:
import matplotlib.pyplot as plt
if history['steps']:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    ax1.plot(history['steps'], history['val_loss'], 'b-o', ms=4)
    ax1.set(xlabel='Step', ylabel='Loss', title='Val Loss'); ax1.grid(True)
    ax2.plot(history['steps'], history['val_tok_acc'], 'g-o', ms=4)
    ax2.set(xlabel='Step', ylabel='Token Accuracy', title='Val Token Acc'); ax2.grid(True)
    plt.tight_layout()
    plt.savefig('results/training_curve.png', dpi=120)
    plt.show()
    print(f'Final val loss: {history["val_loss"][-1]:.4f}')
    print(f'Final val tok_acc: {history["val_tok_acc"][-1]:.4f}')


## Step 11 — Autoregressive generation (greedy decoding)

In [ ]:
from generate import generate_answer
import torch as _torch

ckpt = _torch.load(os.path.join(CHECKPOINT_DIR, 'best_model.pt'), map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded best model (step={ckpt["step"]}  val_loss={ckpt["val_loss"]:.4f})')
_data_source_info = ckpt.get('data_source_info', {})
print(f'Data source used during training: {_data_source_info.get("data_source", "unknown")}')

# Read reverse_answer from the checkpoint itself (never guessed) -- a model
# trained with generated_reverse_answer=True generates answers least-
# significant-digit-first, so raw generate_answer() output must be
# un-reversed before comparing/displaying, exactly like evaluate.py does.
_reverse = bool(_data_source_info.get('configuration', {}).get('reverse_answer', False))
print(f'Configuration: reverse_answer={_reverse}')

print('\n--- Generating answers for 10 test-set questions (never trained on) ---')
for it in test_items[:10]:
    pred, _ = generate_answer(model, tokenizer, it['question'], device, reverse_answer=_reverse)
    ok = '✓' if pred.strip() == it['answer'].strip() else '✗'
    print(f'  {ok}  Q: {it["question"][:60]}')
    print(f'       gold={it["answer"]!r}  pred={pred!r}')


## Step 12 — Full unseen test-set evaluation
This is the number that matters. Every example in `test_items` comes from
the dataset authors' own `interpolate` split — never seen during training.

In [ ]:
from evaluate import full_evaluation

eval_results = full_evaluation(
    checkpoint_path=os.path.join(CHECKPOINT_DIR, 'best_model.pt'),
    data_cfg=data_cfg,
    device_str=DEVICE,
    max_new_tokens=160,
    show_failures=10,
)


## Step 13 — Failure analysis

In [ ]:
failures = eval_results['gen_results'].get('failures', [])
print(f'Failures: {len(failures)}/{eval_results["gen_results"]["total_examples"]}')
for i, f in enumerate(failures[:10]):
    print(f'  [{i+1}] {f["category"]}')
    print(f'       Q   : {f["question"][:70]}')
    print(f'       gold: {f["gold_answer"]!r}')
    print(f'       pred: {f["pred_answer"]!r}')
    print()


## Step 14 — Interactive question mode
Type your own math questions. The model answers using ONLY its trained
weights — no calculator, no external API, no hard-coded answers. Type
`exit` or `quit` to stop.

In [ ]:
from generate import interactive_demo

# Read reverse_answer from the checkpoint loaded in the Step 12 cell above
# (never guessed) so the interactive display un-reverses generated answers
# exactly like evaluate.py's real 95%+ accuracy numbers already account for.
_data_source_info = ckpt.get('data_source_info', {})
_reverse = bool(_data_source_info.get('configuration', {}).get('reverse_answer', False))

interactive_demo(model, tokenizer, device, max_new_tokens=160, temperature=0.0,
                  reverse_answer=_reverse)


## Step 15 — Save to Google Drive

In [ ]:
import os

if USE_DRIVE:
    print(f'Checkpoints were already written directly to Google Drive '
          f'during training (see CHECKPOINT_DIR in Step 2): {CHECKPOINT_DIR}')
    print('Nothing more to copy — they will survive a runtime restart.')
else:
    print('USE_DRIVE was False, so checkpoints only exist on this runtime\'s '
          'local disk and will be LOST on the next restart/disconnect. '
          'Uncomment the block below to copy them to Drive manually.')
    # from google.colab import drive
    # drive.mount('/content/drive')
    # import shutil
    # DRIVE = '/content/drive/MyDrive/math-llm'
    # shutil.copytree(CHECKPOINT_DIR, DRIVE + '/checkpoints', dirs_exist_ok=True)
    # shutil.copytree('results',      DRIVE + '/results',      dirs_exist_ok=True)
    # print(f'Saved to {DRIVE}')

print('checkpoints/:', os.listdir(CHECKPOINT_DIR))
print('results/:',     os.listdir('results'))


## Step 16 — Reload a saved checkpoint and chat with it (no retraining)

Run this cell on its own, any time — even in a **fresh Colab runtime** that
never ran the training cells above. It does not depend on any variable
from earlier cells (`test_items`, `model`, `tokenizer`, ...); it loads
everything it needs straight from the checkpoint file, then drops you into
the same interactive question loop as Step 14.

In [ ]:
import os, torch
from model import MathLLM
from config import ModelConfig
from tokenizer import MathTokenizer
from generate import generate_answer, interactive_demo

CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'best_model.pt')

assert os.path.isfile(CHECKPOINT_PATH), (
    f"No checkpoint found at {CHECKPOINT_PATH!r}. "
    f"If this is a fresh runtime, first run Steps 1-3 above (clone repo, "
    f"install deps, pick device) so the module imports work, then either "
    f"train a model (Steps 4-9) or copy a previously saved checkpoint here."
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt   = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)

reloaded_model = MathLLM(ModelConfig(**ckpt['cfg_model'])).to(device)
reloaded_model.load_state_dict(ckpt['model_state'])
reloaded_model.eval()

tok_path = ckpt.get('tokenizer_path', os.path.join(CHECKPOINT_DIR, 'tokenizer.json'))
reloaded_tokenizer = MathTokenizer.load(tok_path)

# Read reverse_answer straight from what training recorded in the
# checkpoint -- never guessed, never left at a silent default. Without
# this, a model trained with generated_reverse_answer=True (least-
# significant-digit-first) will look like it's failing basic arithmetic
# (e.g. "9 + 9" -> "81") when it actually generated the correct answer in
# its trained format ("18" reversed) and nothing un-reversed it for display.
data_source_info = ckpt.get('data_source_info', {})
reverse_answer = bool(data_source_info.get('configuration', {}).get('reverse_answer', False))

print(f"Reloaded: {ckpt['cfg_model']['name']}  "
      f"step={ckpt['step']}  val_loss={ckpt['val_loss']:.4f}")
print(f"Data source: {data_source_info.get('data_source', 'unknown')}")
print(f"Configuration: reverse_answer={reverse_answer}")

# Quick smoke test with a hand-written question (no dependency on test_items).
# IMPORTANT: use the bare "a OP b" format the generated benchmark actually
# trains on (no "What is ...?" wrapper) -- that phrasing only exists in the
# REAL dataset's more varied templates, so testing it against a
# generated-benchmark checkpoint is an out-of-distribution prompt and will
# produce garbage/empty output, not a sign anything is broken.
smoke_question = "12 + 7"
a, _ = generate_answer(reloaded_model, reloaded_tokenizer, smoke_question, device,
                        max_new_tokens=160, reverse_answer=reverse_answer)
print(f"\nSmoke test: Q={smoke_question!r}  pred={a!r}\n")

# ── Interactive question mode (same as Step 14, using the reloaded model) ──
interactive_demo(reloaded_model, reloaded_tokenizer, device,
                  max_new_tokens=160, temperature=0.0,
                  reverse_answer=reverse_answer)